# Pipeline ETL Unificado — Historial de Compras y Maestro Consolidado

Procesa en memoria los archivos crudos de `compras/*.txt`, filtra compras con entrega efectiva (`cant_entr > 0`), seudonimiza comprobantes con `HMAC-SHA256`, exporta `compras_etl/data/processed/compras.parquet` y genera el maestro consolidado en `archivo_final/maestro_compras.parquet` sin modificar el catálogo original.

## 1. Ingesta en RAM desde `compras/*.txt`

Lee en orden natural los archivos `.txt` de `compras/` (UTF-8 estricto con respaldo `cp1252`), parsea con delimitador `" , "` validando 9 columnas y reporta volumen y rango temporal detectado.

In [13]:
import hashlib
import hmac
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd
import pyarrow as pa
from dotenv import load_dotenv

_cwd = Path.cwd().resolve()
ROOT_DIR = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "compras").is_dir() and (p / "compras_etl").is_dir()),
    _cwd,
)

INPUT_DIR = ROOT_DIR / "compras"
OUTPUT_PATH = ROOT_DIR / "archivo_final" / "maestro_compras.parquet"
CATALOGO_ORIGEN_PATH = ROOT_DIR / "etl" / "data" / "processed" / "catalogo.parquet"
MAESTRO_FINAL_PATH = ROOT_DIR / "archivo_final" / "maestro_productos.parquet"

COLUMNS = [
    "factura",
    "producto",
    "codigo",
    "medio",
    "fecha_hora",
    "cant_ped",
    "cant_fact",
    "cant_entr",
    "precio",
]
TEXT_COLUMNS = ["factura", "producto", "codigo", "medio"]
NUMERIC_INT_COLUMNS = ["cant_ped", "cant_fact", "cant_entr"]

NATURAL_SORT_RE = re.compile(r"(\d+)")
MULTI_SPACE_RE = re.compile(r" {2,}")

SPACE_LIKE_INVISIBLE = {
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002", "\u2003", "\u2004", "\u2005",
    "\u2006", "\u2007", "\u2008", "\u2009", "\u200a", "\u202f", "\u205f", "\u3000",
}
REMOVE_INVISIBLE = {
    "\ufeff", "\u200b", "\u200c", "\u200d", "\u2060", "\u00ad", "\ufffd",
}


def natural_sort_key(path: Path) -> list[int | str]:
    return [
        int(part) if part.isdigit() else part.lower()
        for part in NATURAL_SORT_RE.split(path.name)
    ]


def decode_bytes_strict_or_fallback(raw: bytes) -> tuple[str, bool]:
    if raw.startswith(b"\xef\xbb\xbf"):
        raw = raw[3:]
    try:
        return raw.decode("utf-8", errors="strict"), False
    except UnicodeDecodeError:
        return raw.decode("cp1252", errors="replace"), True


def is_invisible_or_control(ch: str) -> bool:
    code = ord(ch)
    if code < 32 or code == 127 or (128 <= code <= 159):
        return True
    if ch in SPACE_LIKE_INVISIBLE or ch in REMOVE_INVISIBLE:
        return True
    return unicodedata.category(ch) in {"Cc", "Cf", "Zl", "Zp"}


def sanitize_text(value: str) -> tuple[str, int]:
    cleaned_chars: list[str] = []
    removed_or_replaced = 0
    for ch in value:
        if ch in SPACE_LIKE_INVISIBLE:
            cleaned_chars.append(" ")
            removed_or_replaced += 1
        elif is_invisible_or_control(ch):
            removed_or_replaced += 1
        else:
            cleaned_chars.append(ch)
    normalized = "".join(cleaned_chars)
    normalized = normalized.encode("utf-8", errors="ignore").decode("utf-8")
    normalized = MULTI_SPACE_RE.sub(" ", normalized).strip()
    return normalized, removed_or_replaced


def parse_precio_series(series: pd.Series) -> pd.Series:
    def _normalize_price(raw_val: str) -> str:
        val = raw_val.strip().replace("$", "").replace(" ", "")
        if "," in val and "." in val:
            if val.rfind(",") > val.rfind("."):
                val = val.replace(".", "").replace(",", ".")
            else:
                val = val.replace(",", "")
        elif "," in val:
            val = val.replace(",", ".")
        return val

    cleaned = series.astype(str).map(_normalize_price)
    return pd.to_numeric(cleaned, errors="raise").astype("float64")


files = sorted(INPUT_DIR.glob("*.txt"), key=natural_sort_key)
if not files:
    raise FileNotFoundError(f"No se encontraron archivos .txt en {INPUT_DIR.relative_to(ROOT_DIR).as_posix()}")

rows: list[list[str]] = []
load_stats = Counter(files_total=len(files), files_cp1252=0, raw_rows=0)

for file_path in files:
    text, had_fallback = decode_bytes_strict_or_fallback(file_path.read_bytes())
    if had_fallback:
        load_stats["files_cp1252"] += 1

    lines = text.splitlines()
    if not lines:
        continue

    for line_num, line in enumerate(lines[1:], start=2):
        if not line.strip():
            continue
        parts = [p.strip() for p in line.split(" , ")]
        if len(parts) != len(COLUMNS):
            raise ValueError(
                f"Línea malformada en {file_path.name} (línea {line_num}): "
                f"se esperaban {len(COLUMNS)} columnas, se obtuvieron {len(parts)}."
            )
        parts.append(file_path.name)
        rows.append(parts)
        load_stats["raw_rows"] += 1

df_raw = pd.DataFrame(rows, columns=COLUMNS + ["_archivo_origen"])
fechas_detectadas = pd.to_datetime(
    df_raw["fecha_hora"].astype(str).str.strip(),
    format="%Y-%m-%d %H:%M",
    errors="raise",
)

print(f"Carpeta de entrada    : {INPUT_DIR.relative_to(ROOT_DIR).as_posix()}")
print(f"Archivos procesados   : {load_stats['files_total']:,}")
print(f"Archivos con cp1252   : {load_stats['files_cp1252']:,}")
print(f"Filas cargadas        : {len(df_raw):,}")
print(f"Rango de fechas       : {fechas_detectadas.min()} -> {fechas_detectadas.max()}")

Carpeta de entrada    : compras
Archivos procesados   : 70
Archivos con cp1252   : 0
Filas cargadas        : 375,256
Rango de fechas       : 2021-01-01 22:23:00 -> 2026-10-31 10:21:00


## 2. Sanitización de Texto, Tipos y Reglas de Negocio

Limpia caracteres invisibles Unicode, colapsa espacios y normaliza `producto` en mayúsculas, valida cero facturas cruzadas, descarta `medio` y encabezados de combos, y filtra únicamente las líneas con entrega efectiva (`cant_entr > 0`).

In [14]:
df = df_raw.copy()
clean_stats = Counter()

for col in TEXT_COLUMNS:
    series = df[col].astype(str)
    clean_stats[f"multi_spaces_{col}"] += int(series.str.contains(MULTI_SPACE_RE, regex=True).sum())
    cleaned_values: list[str] = []
    invisible_count = 0
    for val in series:
        cleaned, replaced = sanitize_text(val)
        cleaned_values.append(cleaned)
        invisible_count += replaced
    clean_stats[f"invisible_chars_{col}"] += invisible_count
    df[col] = cleaned_values

df["producto"] = df["producto"].astype(str).str.strip().str.upper()
df["factura"] = df["factura"].astype(str).str.strip()
df["codigo"] = df["codigo"].astype(str).str.strip().replace("", pd.NA)
df["medio"] = df["medio"].astype(str).str.strip().replace("", pd.NA)

df["fecha_hora"] = pd.to_datetime(
    df["fecha_hora"].astype(str).str.strip(),
    format="%Y-%m-%d %H:%M",
    errors="raise",
)
for col in NUMERIC_INT_COLUMNS:
    df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="raise").astype("int32")

df["precio"] = parse_precio_series(df["precio"])

cross_file_dups = int((df.groupby("factura")["_archivo_origen"].nunique() > 1).sum())
assert cross_file_dups == 0, f"Se detectaron {cross_file_dups} facturas en más de un archivo."

intra_invoice_promo = int(df.duplicated(subset=COLUMNS + ["_archivo_origen"]).sum())
df = df.drop(columns=["_archivo_origen"])

prefijos_medio_nulo = (
    df.loc[df["medio"].isna(), "factura"]
    .astype(str)
    .str.split("-")
    .str[0]
    .value_counts(dropna=False)
)
n_medio_nulo = int(df["medio"].isna().sum())
if n_medio_nulo > 0:
    assert set(prefijos_medio_nulo.index) == {"O"}, (
        f"Se detectaron valores nulos en 'medio' fuera del circuito 'O': {prefijos_medio_nulo.to_dict()}"
    )

mask_combos = df["codigo"].isna() & (df["cant_ped"] == 0) & (df["cant_fact"] == 0)
n_combos = int(mask_combos.sum())
n_filas_antes = len(df)

df = df.drop(columns=["medio"]).loc[~mask_combos].reset_index(drop=True)
assert (n_filas_antes - len(df)) == n_combos, "La diferencia de filas no coincide con los combos depurados."
assert int(df["codigo"].isna().sum()) == 0, "Persisten filas con 'codigo' vacío tras eliminar combos."
df["codigo"] = df["codigo"].astype(str).str.strip()

mask_sin_entrega = df["cant_entr"] <= 0
n_sin_entrega = int(mask_sin_entrega.sum())
df = df.loc[~mask_sin_entrega].reset_index(drop=True)
assert bool((df["cant_entr"] > 0).all()), "Persisten filas con cant_entr <= 0 tras el filtrado."

total_multi_spaces = sum(v for k, v in clean_stats.items() if k.startswith("multi_spaces_"))
total_invisibles = sum(v for k, v in clean_stats.items() if k.startswith("invisible_chars_"))

print(f"Espacios múltiples corregidos   : {total_multi_spaces:,} (en producto: {clean_stats['multi_spaces_producto']:,})")
print(f"Caracteres invisibles eliminados: {total_invisibles:,}")
print(f"Facturas en >1 archivo          : {cross_file_dups:,} (validado == 0)")
print(f"Repeticiones intra-factura      : {intra_invoice_promo:,} (válidas por topes promocionales; conservadas)")
print(f"Redundancia de 'medio'          : {n_medio_nulo:,} nulos -> 100% circuito {list(prefijos_medio_nulo.index)} (columna eliminada)")
print(f"Encabezados de combos sin EAN   : {n_combos:,} filas eliminadas")
print(f"Filas sin entrega (cant_entr=0) : {n_sin_entrega:,} filas eliminadas")
print(f"Filas efectivas resultantes     : {len(df):,}")

Espacios múltiples corregidos   : 79,304 (en producto: 79,304)
Caracteres invisibles eliminados: 0
Facturas en >1 archivo          : 0 (validado == 0)
Repeticiones intra-factura      : 22 (válidas por topes promocionales; conservadas)
Redundancia de 'medio'          : 698 nulos -> 100% circuito ['O'] (columna eliminada)
Encabezados de combos sin EAN   : 56 filas eliminadas
Filas sin entrega (cant_entr=0) : 297,426 filas eliminadas
Filas efectivas resultantes     : 77,774


## 3. Seudonimización Determinística con `HMAC-SHA256` + `SALT`

Carga `FACTURA_HASH_SALT` desde `.env` mediante `python-dotenv` y reemplaza cada factura única por `<PREFIJO>-<HMAC_SHA256[:10].upper()>`, verificando ausencia de colisiones.

In [15]:
env_path = ROOT_DIR / ".env"
load_dotenv(env_path, override=True)
salt = os.environ.get("FACTURA_HASH_SALT", "").strip()
if not salt:
    raise RuntimeError("FACTURA_HASH_SALT no está configurado o está vacío en .env")

salt_bytes = salt.encode("utf-8")


def pseudonymize_factura(factura_real: str) -> str:
    if "-" not in factura_real:
        raise ValueError("Formato de comprobante sin guion separador de prefijo.")
    prefijo = factura_real.split("-", 1)[0].strip().upper()
    if not re.fullmatch(r"[A-Z0-9]+", prefijo):
        raise ValueError(f"Prefijo de circuito inválido: {prefijo!r}")
    digest = hmac.new(salt_bytes, factura_real.encode("utf-8"), hashlib.sha256).hexdigest()[:10].upper()
    return f"{prefijo}-{digest}"


facturas_unicas = df["factura"].astype(str).unique()
n_facturas_antes = len(facturas_unicas)
factura_map = {fac: pseudonymize_factura(fac) for fac in facturas_unicas}

df["factura"] = df["factura"].astype(str).map(factura_map)
assert int(df["factura"].isna().sum()) == 0, "Quedaron facturas sin seudonimizar."

n_facturas_despues = int(df["factura"].nunique())
assert n_facturas_antes == n_facturas_despues, (
    f"Colisión detectada en seudonimización: {n_facturas_antes} != {n_facturas_despues}"
)

resumen_prefijos = (
    df.assign(_prefijo=df["factura"].str.split("-").str[0])
    .groupby("_prefijo")
    .agg(
        comprobantes_unicos=("factura", "nunique"),
        lineas=("factura", "size"),
    )
    .rename_axis("prefijo")
    .sort_values("comprobantes_unicos", ascending=False)
)

print(f"Comprobantes únicos seudonimizados: {n_facturas_despues:,} (cero colisiones verificadas)")
print("\nResumen agregado por prefijo de circuito:")
print(resumen_prefijos.to_string())

Comprobantes únicos seudonimizados: 19,068 (cero colisiones verificadas)

Resumen agregado por prefijo de circuito:
         comprobantes_unicos  lineas
prefijo                             
S                      15828   70210
T                       2908    6400
O                        326    1156
F                          6       8


## 4. Saneamiento Financiero y Reconciliación

Calcula `precio_unitario` e `importe_facturado` sin doble redondeo, reporta métricas de auditoría en memoria, elimina la columna cruda `precio` y ordena cronológicamente.

In [16]:
suma_precio_pedido = float(df["precio"].sum())

denom = df["cant_ped"].where(df["cant_ped"] > 0, df["cant_fact"])
precio_unit_exacto = df["precio"].div(denom).where(denom > 0, 0.0)
df["precio_unitario"] = precio_unit_exacto.round(2)
df["importe_facturado"] = (
    df["precio"]
    .where(df["cant_fact"] == df["cant_ped"], precio_unit_exacto * df["cant_fact"])
    .round(2)
)

unid_bonificadas = (df["cant_entr"] - df["cant_fact"]).clip(lower=0)
n_lineas_bonificadas = int((unid_bonificadas > 0).sum())
total_unid_bonificadas = int(unid_bonificadas.sum())

df = (
    df.drop(columns=["precio"])
    .sort_values(["fecha_hora", "factura", "codigo"], kind="mergesort")
    .reset_index(drop=True)
)

gasto_efectivo = float(df["importe_facturado"].sum())
diferencia_parcial = suma_precio_pedido - gasto_efectivo

print(f"Suma 'precio' sobre líneas entr.  : {suma_precio_pedido:,.2f}")
print(f"Gasto efectivo ('importe_facturado'): {gasto_efectivo:,.2f} ({len(df):,} líneas)")
print(f"Ajuste por entregas parciales/bonif.: {diferencia_parcial:,.2f}")
print(f"Líneas con unidades bonificadas     : {n_lineas_bonificadas:,} ({total_unid_bonificadas:,} unidades)")

Suma 'precio' sobre líneas entr.  : 568,042,827.99
Gasto efectivo ('importe_facturado'): 566,760,205.92 (77,774 líneas)
Ajuste por entregas parciales/bonif.: 1,282,622.07
Líneas con unidades bonificadas     : 54 (466 unidades)


## 5. Exportación Única a Parquet y Certificación (`assert`)

Exporta el dataset final a `compras_etl/data/processed/compras.parquet` (Snappy), valida integridad tras recargarlo con `pd.read_parquet` y muestra el peso y 10 filas.

In [17]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False, compression="snappy")

size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
df_check = pd.read_parquet(OUTPUT_PATH)

assert df_check.shape == df.shape, f"Dimensiones distintas: {df_check.shape} != {df.shape}"
assert list(df_check.columns) == list(df.columns), f"Columnas distintas: {list(df_check.columns)}"
assert bool((df_check["cant_entr"] > 0).all()), "Existen registros con cant_entr <= 0 en el Parquet."

cols_clave = ["factura", "codigo", "fecha_hora"]
nulos_clave = df_check[cols_clave].isna().sum()
assert int(nulos_clave.sum()) == 0, f"Nulos detectados en columnas clave: {nulos_clave.to_dict()}"

patron_hmac = r"^[A-Z0-9]+-[A-F0-9]{10}$"
assert bool(df_check["factura"].astype(str).str.fullmatch(patron_hmac).all()), (
    "Se detectaron comprobantes que no cumplen el patrón HMAC esperado."
)

for col in ["factura", "producto", "codigo"]:
    for val in df_check[col].astype(str):
        val.encode("utf-8", errors="strict")
        assert not any(is_invisible_or_control(ch) for ch in val), f"Carácter invisible residual en {col}"

assert size_mb < 50, f"El archivo Parquet supera el límite de 50 MB: {size_mb:.2f} MB"

print("=== CERTIFICACIÓN PARQUET COMPRAS EXITOSA ===")
print(f"Archivo exportado : {OUTPUT_PATH.relative_to(ROOT_DIR).as_posix()}")
print(f"Dimensiones       : {df_check.shape[0]:,} filas x {df_check.shape[1]} columnas")
print(f"Tamaño en disco   : {size_mb:.2f} MB (< 50 MB)")
print("\nMuestra de 10 filas efectivas:")
print(df_check.head(10).to_string())

=== CERTIFICACIÓN PARQUET COMPRAS EXITOSA ===
Archivo exportado : archivo_final/maestro_compras.parquet
Dimensiones       : 77,774 filas x 9 columnas
Tamaño en disco   : 1.69 MB (< 50 MB)

Muestra de 10 filas efectivas:
        factura                       producto         codigo          fecha_hora  cant_ped  cant_fact  cant_entr  precio_unitario  importe_facturado
0  S-9C67A2D05A            BAYASPIRINA COM X30  7793640000747 2021-01-01 22:23:00         3          3          3           144.23             432.70
1  S-9C67A2D05A                SINCERUM GTS X5  7795331006670 2021-01-01 22:23:00         2          2          2           278.96             557.92
2  S-9C67A2D05A                  GINKAN OV X12  7795338001258 2021-01-01 22:23:00         1          1          1          1001.49            1001.49
3  S-9C67A2D05A             CIRIAX COM 500 X20  7795345002644 2021-01-01 22:23:00         1          1          1          1182.62            1182.62
4  S-9C67A2D05A          EVATE

## 6. Generación del Maestro Consolidado (`archivo_final/maestro_compras.parquet`)

Lee `etl/data/processed/catalogo.parquet` en modo solo lectura (sin modificarlo), incorpora los SKUs huérfanos detectados en compras usando únicamente `codigo` y su última descripción de factura (`producto`), y guarda el maestro definitivo en `archivo_final/maestro_compras.parquet`.

In [18]:
def generar_maestro_compras(
    df_compras: pd.DataFrame,
    catalogo_origen_path: Path,
    maestro_destino_path: Path,
) -> pd.DataFrame:
    """
    Genera archivo_final/maestro_compras.parquet combinando el catálogo maestro original
    (sin modificarlo) con los SKUs presentes en compras que no existían en el catálogo,
    poblando únicamente codigo y producto/descripcion desde las facturas.
    """
    if not catalogo_origen_path.is_file():
        raise FileNotFoundError(
            f"No se encontró el catálogo origen en {catalogo_origen_path.relative_to(ROOT_DIR).as_posix()}"
        )

    df_cat = pd.read_parquet(catalogo_origen_path)
    if "codigo" not in df_cat.columns:
        raise ValueError(f"El catálogo no contiene la columna 'codigo'. Columnas: {list(df_cat.columns)}")

    df_cat["codigo"] = df_cat["codigo"].astype(str).str.strip()
    assert int(df_cat["codigo"].duplicated().sum()) == 0, "El catálogo origen contiene códigos duplicados."

    candidatos_desc = ["producto", "descripcion", "nombre", "producto_nombre", "detalle"]
    col_desc_cat = next((c for c in candidatos_desc if c in df_cat.columns), None)
    if col_desc_cat is None:
        raise ValueError(
            f"No se encontró columna de descripción en el catálogo. Columnas: {list(df_cat.columns)}"
        )

    skus_compras = set(df_compras["codigo"].astype(str).str.strip())
    skus_catalogo = set(df_cat["codigo"])
    skus_huerfanos = sorted(skus_compras - skus_catalogo)

    print(f"Catálogo origen (solo lect.): {catalogo_origen_path.relative_to(ROOT_DIR).as_posix()}")
    print(f"SKUs en catálogo origen     : {len(df_cat):,}")
    print(f"SKUs únicos en compras      : {len(skus_compras):,}")
    print(f"SKUs huérfanos a incorporar : {len(skus_huerfanos):,}")

    if skus_huerfanos:
        df_huerfanos_factura = (
            df_compras.loc[df_compras["codigo"].isin(skus_huerfanos)]
            .sort_values("fecha_hora", kind="mergesort")
            .groupby("codigo", as_index=False)
            .agg(_desc_factura=("producto", "last"))
        )

        df_nuevos = pd.DataFrame(index=range(len(df_huerfanos_factura)), columns=df_cat.columns)
        df_nuevos["codigo"] = df_huerfanos_factura["codigo"].values
        df_nuevos[col_desc_cat] = df_huerfanos_factura["_desc_factura"].values

        df_maestro = pd.concat([df_cat, df_nuevos], ignore_index=True)
    else:
        df_maestro = df_cat.copy()

    maestro_destino_path.parent.mkdir(parents=True, exist_ok=True)
    df_maestro.to_parquet(maestro_destino_path, index=False, compression="snappy")

    df_maestro_check = pd.read_parquet(maestro_destino_path)
    size_maestro_mb = maestro_destino_path.stat().st_size / (1024 * 1024)

    assert df_maestro_check.shape == df_maestro.shape, "Dimensiones distintas tras recargar maestro_compras.parquet"
    assert int(df_maestro_check["codigo"].isna().sum()) == 0, "Nulos en 'codigo' dentro de maestro_compras.parquet"
    assert int(df_maestro_check["codigo"].duplicated().sum()) == 0, "Códigos duplicados en maestro_compras.parquet"
    assert skus_compras.issubset(set(df_maestro_check["codigo"])), (
        "Persisten SKUs de compras sin correspondencia en maestro_compras.parquet"
    )
    assert size_maestro_mb < 50, f"El archivo supera 50 MB: {size_maestro_mb:.2f} MB"

    print("\n=== CERTIFICACIÓN MAESTRO DEFINITIVO EXITOSA ===")
    print(f"Archivo generado            : {maestro_destino_path.relative_to(ROOT_DIR).as_posix()}")
    print(f"Dimensiones                 : {df_maestro_check.shape[0]:,} filas x {df_maestro_check.shape[1]} columnas")
    print(f"Tamaño en disco             : {size_maestro_mb:.2f} MB (< 50 MB)")
    print(f"Integridad referencial      : 100.00% (campos poblados en huérfanos: ['codigo', '{col_desc_cat}'])")
    return df_maestro_check


df_maestro_compras = generar_maestro_compras(df_check, CATALOGO_ORIGEN_PATH, MAESTRO_FINAL_PATH)
print("\nMuestra de últimos 5 registros incorporados:")
print(df_maestro_compras.tail(5).to_string())

Catálogo origen (solo lect.): etl/data/processed/catalogo.parquet
SKUs en catálogo origen     : 30,326
SKUs únicos en compras      : 5,204
SKUs huérfanos a incorporar : 758

=== CERTIFICACIÓN MAESTRO DEFINITIVO EXITOSA ===
Archivo generado            : archivo_final/maestro_productos.parquet
Dimensiones                 : 31,084 filas x 7 columnas
Tamaño en disco             : 0.78 MB (< 50 MB)
Integridad referencial      : 100.00% (campos poblados en huérfanos: ['codigo', 'producto'])

Muestra de últimos 5 registros incorporados:
              codigo                                  producto droga tipo accion clasificacion laboratorio
31079  8710103876564        AVENT MAMAD 262/06 ENTREN NAT X125   NaN  NaN    NaN           NaN         NaN
31080  8710103881209      AVENT TETINA 635-SILX2CLASSIC 3-ORIF   NaN  NaN    NaN           NaN         NaN
31081  8710103938422  AVENT CHUP 222/01 U.SOFT DECO NENE0A6 X2   NaN  NaN    NaN           NaN         NaN
31082  8710103942702  AVENT CHUP 085